# Algorithm 2.1 — Probability Tables

*Kochenderfer, Wheeler, Wray — Algorithms for Decision Making, Chapter 2: Representation*

Before we can build Bayesian networks (2.2) or compute joint probabilities via
the chain rule (2.3), we need a way to represent a discrete probability
distribution in code. That's what Algorithm 2.1 is: the data structures for
**discrete variables** and **factors** (probability tables).

This notebook covers:
1. Discrete variables
2. Assignments
3. Factor tables (the probability table itself)
4. Dense array representation (an equivalent, more compact view)
5. Synthetic data: sampling from a factor
6. Saving the data in a memory-efficient format


## 1. Discrete variables

The book represents a discrete random variable as just a **name** plus **how
many values it can take**, `r`. The values themselves are just integers
`0, 1, ..., r-1` — any real-world meaning (e.g. `0 = "no rain"`,
`1 = "rain"`) is kept separately, outside the core algorithm.


In [1]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Variable:
    """A discrete random variable with `r` possible values (0..r-1)."""
    name: str
    r: int  # number of values the variable can take


# Three binary variables for our running example.
rain = Variable("rain", 2)      # 0 = no rain,   1 = rain
traffic = Variable("traffic", 2)  # 0 = light,     1 = heavy
late = Variable("late", 2)      # 0 = on time,   1 = late

variables = [rain, traffic, late]

# Human-readable labels for each value, kept separate from the algorithm itself.
labels = {
    "rain": ["no rain", "rain"],
    "traffic": ["light", "heavy"],
    "late": ["on time", "late"],
}

for v in variables:
    print(f"{v.name}: r={v.r}, values={labels[v.name]}")


rain: r=2, values=['no rain', 'rain']
traffic: r=2, values=['light', 'heavy']
late: r=2, values=['on time', 'late']


## 2. Assignments

An **assignment** maps each variable to one of its values. In Julia the book
uses a `Dict{Symbol,Int}` (or a `NamedTuple`) for this. In Python, a plain
`dict` isn't hashable, so we represent an assignment as a **sorted tuple of
`(name, value)` pairs** — that keeps it usable as a dictionary key later.


In [2]:
from itertools import product
from typing import Tuple

Assignment = Tuple[Tuple[str, int], ...]


def assign(**kwargs) -> Assignment:
    """Build a hashable assignment from keyword values, e.g. assign(rain=1, traffic=0)."""
    return tuple(sorted(kwargs.items()))


def all_assignments(variables) -> list:
    """Every possible combination of values for a list of Variables."""
    domains = [range(v.r) for v in variables]
    return [tuple(sorted(zip((v.name for v in variables), values)))
            for values in product(*domains)]


example = assign(rain=1, traffic=0, late=0)
print("example assignment:", example)

full_domain = all_assignments(variables)
print(f"\n{len(full_domain)} possible assignments over {[v.name for v in variables]}:")
for a in full_domain:
    print(" ", a)


example assignment: (('late', 0), ('rain', 1), ('traffic', 0))

8 possible assignments over ['rain', 'traffic', 'late']:
  (('late', 0), ('rain', 0), ('traffic', 0))
  (('late', 1), ('rain', 0), ('traffic', 0))
  (('late', 0), ('rain', 0), ('traffic', 1))
  (('late', 1), ('rain', 0), ('traffic', 1))
  (('late', 0), ('rain', 1), ('traffic', 0))
  (('late', 1), ('rain', 1), ('traffic', 0))
  (('late', 0), ('rain', 1), ('traffic', 1))
  (('late', 1), ('rain', 1), ('traffic', 1))


## 3. Factor tables

A **factor table** (`FactorTable`) maps every assignment to a probability.
A **factor** (`Factor`) is just the pair of `(variables, table)`. For a
*joint* distribution over all variables, the table must sum to 1 across all
assignments — that's the only real constraint.

Here we hand-write a joint distribution over `rain`, `traffic`, `late` that
tells a small story: rain makes heavy traffic more likely, and heavy traffic
makes being late more likely.


In [3]:
@dataclass(frozen=True)
class Factor:
    """A factor: a set of variables plus a table mapping assignments to probabilities."""
    variables: list
    table: dict  # Assignment -> float


# P(rain, traffic, late), built by hand so the story is easy to follow.
# Format: (rain, traffic, late) -> probability
raw_table = {
    (0, 0, 0): 0.35,  # no rain, light traffic, on time
    (0, 0, 1): 0.05,  # no rain, light traffic, late (rare - other reasons)
    (0, 1, 0): 0.10,  # no rain, heavy traffic, on time
    (0, 1, 1): 0.10,  # no rain, heavy traffic, late
    (1, 0, 0): 0.05,  # rain, light traffic, on time
    (1, 0, 1): 0.03,  # rain, light traffic, late
    (1, 1, 0): 0.07,  # rain, heavy traffic, on time
    (1, 1, 1): 0.25,  # rain, heavy traffic, late
}

names = [v.name for v in variables]
table = {
    tuple(sorted(zip(names, values))): p
    for values, p in raw_table.items()
}

rain_traffic_late = Factor(variables=variables, table=table)

total = sum(rain_traffic_late.table.values())
print(f"table has {len(rain_traffic_late.table)} entries, sums to {total:.4f}")
assert abs(total - 1.0) < 1e-9, "a joint probability table must sum to 1"

print("\nP(rain, traffic, late):")
for a, p in rain_traffic_late.table.items():
    readable = ", ".join(f"{k}={labels[k][v]}" for k, v in a)
    print(f"  {readable:45s} {p:.2f}")


table has 8 entries, sums to 1.0000

P(rain, traffic, late):
  late=on time, rain=no rain, traffic=light     0.35
  late=late, rain=no rain, traffic=light        0.05
  late=on time, rain=no rain, traffic=heavy     0.10
  late=late, rain=no rain, traffic=heavy        0.10
  late=on time, rain=rain, traffic=light        0.05
  late=late, rain=rain, traffic=light           0.03
  late=on time, rain=rain, traffic=heavy        0.07
  late=late, rain=rain, traffic=heavy           0.25


## 4. Dense array representation

A `dict`-based `FactorTable` is convenient and matches the book's
pseudocode, but for a *complete* table (every assignment present, as here)
a plain `numpy` array indexed by value is more compact and much faster to
work with. This is the representation we'll actually save to disk.


In [4]:
import numpy as np


def factor_to_array(factor: Factor) -> np.ndarray:
    """Convert a dict-based Factor into a dense ndarray, shape = (r_1, ..., r_k)."""
    shape = tuple(v.r for v in factor.variables)
    arr = np.zeros(shape, dtype=np.float64)
    for assignment, p in factor.table.items():
        values = dict(assignment)
        idx = tuple(values[v.name] for v in factor.variables)
        arr[idx] = p
    return arr


rain_traffic_late_arr = factor_to_array(rain_traffic_late)
print("shape:", rain_traffic_late_arr.shape)
print(rain_traffic_late_arr)
print("\nsum:", rain_traffic_late_arr.sum())


shape: (2, 2, 2)
[[[0.35 0.05]
  [0.1  0.1 ]]

 [[0.05 0.03]
  [0.07 0.25]]]

sum: 1.0


## 5. Synthetic data: sampling from the factor

A factor table over *all* variables is just a joint distribution, so we can
treat it as one to draw samples: flatten the table into a list of
`(assignment, probability)` pairs and sample assignments with those weights.

Per this project's convention, the generator takes a `seed` (default `42`)
so the data is reproducible without needing to regenerate or commit it.


In [5]:
def sample_assignments(factor: Factor, n: int, seed: int = 42) -> np.ndarray:
    """Draw `n` samples from a joint Factor. Returns an (n, k) int8 array of value indices."""
    rng = np.random.default_rng(seed)
    assignments = list(factor.table.keys())
    probs = np.array(list(factor.table.values()))

    choice_idx = rng.choice(len(assignments), size=n, p=probs)
    values_by_var = {v.name: [] for v in factor.variables}
    for i in choice_idx:
        for name, value in assignments[i]:
            values_by_var[name].append(value)

    columns = [values_by_var[v.name] for v in factor.variables]
    return np.array(columns, dtype=np.int8).T  # shape (n, num_variables)


samples = sample_assignments(rain_traffic_late, n=1000, seed=42)
print("samples shape:", samples.shape, "dtype:", samples.dtype)
print("first 5 rows (rain, traffic, late):")
print(samples[:5])

# Sanity check: empirical frequencies should be close to the true table.
empirical = samples[(samples[:, 0] == 1) & (samples[:, 1] == 1) & (samples[:, 2] == 1)].shape[0] / 1000
print(f"\nP(rain=1, traffic=1, late=1) true={rain_traffic_late_arr[1,1,1]:.2f}  empirical={empirical:.3f}")


samples shape: (1000, 3) dtype: int8
first 5 rows (rain, traffic, late):
[[1 1 1]
 [0 1 0]
 [1 1 1]
 [1 1 0]
 [0 0 0]]

P(rain=1, traffic=1, late=1) true=0.25  empirical=0.257


## 6. Saving the data efficiently

For small dense tables like this one, `numpy`'s **compressed** `.npz`
format is the most memory-efficient choice available without adding new
dependencies (`numpy` is already a project dependency; `pandas`/`pyarrow`
are not): it's a binary, typed, columnar-friendly format that compresses
well and loads directly back into arrays with no parsing step.

We save two things to `../data/`:
- `rain_traffic_late_factor.npz` — the dense probability tensor + variable
  metadata (names, domain sizes, labels)
- `rain_traffic_late_samples.npz` — the 1000 synthetic samples (seed=42)

To make the "memory efficient" claim concrete, we compare the compressed
`.npz` size against the same data written out as plain-text JSON.


In [6]:
import os
import json as _json

data_dir = os.path.join("..", "data")
os.makedirs(data_dir, exist_ok=True)

factor_path = os.path.join(data_dir, "rain_traffic_late_factor.npz")
samples_path = os.path.join(data_dir, "rain_traffic_late_samples.npz")

np.savez_compressed(
    factor_path,
    table=rain_traffic_late_arr,
    variable_names=np.array(names),
    variable_sizes=np.array([v.r for v in variables]),
)

np.savez_compressed(
    samples_path,
    samples=samples,
    variable_names=np.array(names),
    seed=np.array(42),
)

npz_bytes = os.path.getsize(factor_path) + os.path.getsize(samples_path)

# Same data, written as plain JSON, for comparison.
json_equivalent = {
    "table": rain_traffic_late_arr.tolist(),
    "variable_names": names,
    "samples": samples.tolist(),
}
json_bytes = len(_json.dumps(json_equivalent).encode("utf-8"))

print(f"compressed .npz total: {npz_bytes:,} bytes")
print(f"equivalent .json:      {json_bytes:,} bytes")
print(f"npz is {json_bytes / npz_bytes:.1f}x smaller")


compressed .npz total: 1,863 bytes
equivalent .json:      11,129 bytes
npz is 6.0x smaller


## 7. Loading it back

Confirming the round trip: load both files and check they match what we
generated above.


In [7]:
with np.load(factor_path) as f:
    loaded_table = f["table"]
    loaded_names = list(f["variable_names"])

with np.load(samples_path) as f:
    loaded_samples = f["samples"]
    loaded_seed = int(f["seed"])

assert np.array_equal(loaded_table, rain_traffic_late_arr)
assert np.array_equal(loaded_samples, samples)
assert loaded_names == names

print("round-trip OK")
print("loaded table shape:", loaded_table.shape)
print("loaded samples shape:", loaded_samples.shape, "seed:", loaded_seed)


round-trip OK
loaded table shape: (2, 2, 2)
loaded samples shape: (1000, 3) seed: 42


## Summary

- A **discrete variable** is just a name plus a count of possible values.
- An **assignment** maps variables to specific values (represented here as a
  sorted, hashable tuple).
- A **factor table** maps assignments to probabilities; a **factor** pairs a
  table with the variables it's over. When it covers *all* variables and
  sums to 1, it's a joint distribution.
- A dense `numpy` array is an equivalent, more efficient representation for
  complete tables, and compresses well for storage (`.npz`).

Next: **2.2** combines multiple factors like this one with a graph structure
to build a full Bayesian network, and **2.3** uses the chain rule to compute
a joint probability as a product of such factors.
